# Explore ScanNet++ `segments_anno.json`

Use this notebook to inspect the ScanNet++ scene-instance annotations used by the PAG target selector.

Recommended kernel: a Python env with `numpy` and `trimesh`, for example `sam3d-objects`.


In [1]:
from pathlib import Path
import json
from pprint import pprint
from collections import Counter

REPO_ROOT = Path('/my_workspace/4DHHOI')
SCANNET_ROOT = REPO_ROOT / 'Scannet++' / 'data'
scene_id = '2b1dc6d6a5'

scene_scan_dir = SCANNET_ROOT / scene_id / 'scans'
segments_anno_path = scene_scan_dir / 'segments_anno.json'
segments_path = scene_scan_dir / 'segments.json'
mesh_path = scene_scan_dir / 'mesh_aligned_0.05.ply'

print('scene_scan_dir =', scene_scan_dir)
print('segments_anno_path exists =', segments_anno_path.exists())
print('segments_path exists =', segments_path.exists())
print('mesh_path exists =', mesh_path.exists())


scene_scan_dir = /my_workspace/4DHHOI/Scannet++/data/2b1dc6d6a5/scans
segments_anno_path exists = True
segments_path exists = True
mesh_path exists = True


In [2]:
with segments_anno_path.open('r', encoding='utf-8') as f:
    segments_anno = json.load(f)

print('Top-level keys:')
print(list(segments_anno.keys()))
print()
print('sceneId =', segments_anno['sceneId'])
print('num segGroups =', len(segments_anno['segGroups']))

first_group = segments_anno['segGroups'][0]
print('\nKeys for one segGroup:')
print(list(first_group.keys()))


Top-level keys:
['sceneId', 'appId', 'annId', 'segGroups']

sceneId = scannetpp.2022-12-04_15-52
num segGroups = 96

Keys for one segGroup:
['id', 'objectId', 'label', 'segments', 'obb', 'dominantNormal', 'partId', 'index']


In [3]:
# Show one object example. Change the label filter if you want a different object class.
example_group = next(g for g in segments_anno['segGroups'] if g['label'] in ('chair', 'office chair'))

example_preview = {
    'id': example_group['id'],
    'objectId': example_group['objectId'],
    'label': example_group['label'],
    'partId': example_group['partId'],
    'index': example_group['index'],
    'num_segments': len(example_group['segments']),
    'segments_head': example_group['segments'][:20],
    'obb': {
        'centroid': example_group['obb']['centroid'],
        'axesLengths': example_group['obb']['axesLengths'],
        'min': example_group['obb']['min'],
        'max': example_group['obb']['max'],
    },
}

pprint(example_preview)


{'id': 8,
 'index': 65,
 'label': 'office chair',
 'num_segments': 7700,
 'obb': {'axesLengths': [0.6047056104977615,
                         0.5798594057559967,
                         0.570715346812378],
         'centroid': [3.956053239395863,
                      3.0831170402106522,
                      0.6980655044317247],
         'max': [4.324495555774168, 3.4398136900823353, 0.9879952073097231],
         'min': [3.587610923017558, 2.726420390338969, 0.40813580155372625]},
 'objectId': 8,
 'partId': 8,
 'segments_head': [134401,
                   134402,
                   134403,
                   134404,
                   134405,
                   134406,
                   134407,
                   134408,
                   134409,
                   134410,
                   134411,
                   134412,
                   134413,
                   134414,
                   134415,
                   134416,
                   134417,
                   134

In [4]:
# Count labels in the scene.
label_counts = Counter(g['label'] for g in segments_anno['segGroups'])
print('Most common labels:')
for label, count in label_counts.most_common(20):
    print(f'{label:20s} {count}')


Most common labels:
object               13
box                  13
wall                 6
monitor              6
power socket         5
ceiling light        4
bag                  3
window               3
bottle               3
mouse                3
keyboard             3
heater               2
light switch         2
table                2
ceiling              2
office chair         2
chair                2
storage cabinet      2
fan                  1
window frame         1


In [5]:
# Inspect the relationship between segments.json and segments_anno.json.
with segments_path.open('r', encoding='utf-8') as f:
    segments_payload = json.load(f)

seg_indices = segments_payload['segIndices']
print('segments.json keys =', list(segments_payload.keys()))
print('num segIndices =', len(seg_indices))
print('first 20 segIndices =', seg_indices[:20])
print('last 10 segIndices =', seg_indices[-10:])
print('looks like identity mapping =', seg_indices[:10] == list(range(10)) and seg_indices[-1] == len(seg_indices) - 1)

same_part_object = sum(1 for g in segments_anno['segGroups'] if g['partId'] == g['objectId'])
print('partId == objectId for all segGroups =', same_part_object == len(segments_anno['segGroups']))


segments.json keys = ['sceneId:', 'segIndices']
num segIndices = 1204456
first 20 segIndices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
last 10 segIndices = [1204446, 1204447, 1204448, 1204449, 1204450, 1204451, 1204452, 1204453, 1204454, 1204455]
looks like identity mapping = True
partId == objectId for all segGroups = True


In [6]:
# Optional: estimate how many mesh vertices/faces belong to the selected example object.
import numpy as np
import trimesh

mesh = trimesh.load(str(mesh_path), force='mesh')
verts = np.asarray(mesh.vertices)
faces = np.asarray(mesh.faces)

# In the downloaded scenes we inspected, example_group['segments'] behaves like vertex ids.
vertex_ids = np.asarray(example_group['segments'], dtype=np.int64)
vertex_mask = np.zeros(len(verts), dtype=bool)
vertex_mask[vertex_ids] = True
face_mask = np.all(vertex_mask[faces], axis=1)

print('mesh vertices =', len(verts))
print('mesh faces =', len(faces))
print('object vertices =', int(vertex_mask.sum()))
print('object faces =', int(face_mask.sum()))


mesh vertices = 1204456
mesh faces = 2491612
object vertices = 7700
object faces = 15955
